In [6]:
#!/usr/bin/env python
"""
Run dialogue_5: therapist emotion-aware (online text VA).
Client = LLM, Therapist = LLM ที่เห็น VA จากข้อความของ client แต่ละเทิร์น
"""

import os
import json
import getpass
from typing import Tuple
from openai import OpenAI

MODEL = "gpt-4o-mini"   # เปลี่ยนรุ่นได้ตามใจ

# ==============================
# 1) OpenAI client
# ==============================
def setup_client() -> OpenAI:
    # ล้างค่าเก่าใน env ถ้ามี
    if "OPENAI_API_KEY" in os.environ:
        del os.environ["OPENAI_API_KEY"]

    # บังคับถาม key ใหม่ทุกครั้งที่รันสคริปต์
    key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = key

    return OpenAI()


client = setup_client()

# ==============================
# 2) ฟังก์ชันหา VA จาก text (ใช้ RobroKools/vad-bert)
# ==============================
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Tuple

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "RobroKools/vad-bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

# ช่วงค่า VAD ดิบของโมเดล (ตาม paper/ตัวอย่างของ vad-bert)
V_MIN, V_MAX = 1.0, 5.0
A_MIN, A_MAX = 1.0, 5.0

def _to_minus1_1(x: float, xmin: float = 1.0, xmax: float = 5.0) -> float:
    """แมปสเกล [xmin, xmax] -> [-1, 1]"""
    return float(2 * (x - xmin) / (xmax - xmin) - 1.0)

def get_text_VA(text: str) -> Tuple[float, float]:
    """
    รับข้อความเดียว แล้วคืนค่า (valence, arousal) แบบ normalized อยู่ในช่วง [-1, 1]
    จากโมเดล RobroKools/vad-bert
    """
    enc = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)

    # logits: [1, 3] = [V, A, D]
    vad = out.logits.cpu().numpy()[0]  # shape (3,)
    v_raw, a_raw, d_raw = vad.tolist()

    v_norm = _to_minus1_1(v_raw, V_MIN, V_MAX)
    a_norm = _to_minus1_1(a_raw, A_MIN, A_MAX)

    return v_norm, a_norm

# ==============================
# 3) System prompt: client & therapist
# ==============================

CLIENT_SYSTEM = """
You are a CBT therapy client talking to therapist "Luna".

- You struggle with anxiety, guilt, and loneliness in your life.
- You sometimes feel misunderstood or skeptical about therapy.
- When the therapist suggests reframing, advice, or homework,
  you may partially resist, question it, or bring up obstacles
  (e.g., "I don't think that will work for me", "It's hard because ...").
- Speak in a natural, first-person voice.
- Stay emotionally consistent across turns.
- Describe thoughts, feelings, and situations in 2–4 sentences per turn.
- In each full dialogue, you must focus on only ONE life problem scenario.
- Do not mix multiple problem seeds in the same dialogue.
- Once a problem seed is assigned for a dialogue, keep that same core life problem throughout the whole dialogue.
"""

CLIENT_USER_TEMPLATE_FIRST = """
Start the first message to your therapist.

Describe what has been bothering you lately (2–4 sentences).
You may already feel unsure whether therapy can really help.

Important:
- This dialogue has exactly ONE assigned life problem scenario.
- You must only use the following scenario in this whole dialogue.
- Do not introduce a second major life problem.

Assigned life problem scenario:
{problem_seed}
"""

CLIENT_USER_TEMPLATE_NEXT = """
Therapist just said:
"{therapist_text}"

Continue the conversation as the client.
Describe what you think and feel now in 2–4 sentences.
If the therapist gives advice, interpretations, or homework,
you can question it, express doubts, or explain why it feels difficult.

Important:
- Stay within the same assigned life problem scenario for this whole dialogue.
- Do not switch to a new major life problem.
"""

PROBLEM_SEEDS = [
    "You are mainly worried about chronic work stress and fear of failure.",
    "You feel intense loneliness after a recent breakup.",
    "You feel guilty about not being a good enough child to your parents.",
    "You are anxious about your future career and financial stability.",
    "You feel social anxiety and avoid meeting people.",
    "You feel guilty and ashamed about a past mistake in a relationship.",
    "You are overwhelmed caring for a sick family member.",
    "You feel stuck and unmotivated in your studies.",
    "You feel like a burden to your friends and family.",
    "You feel anxious about your health and possible illness.",
]

THERAPIST_SYSTEM_EMO = """
You are "Luna", a warm CBT therapist.

You receive for each client message:
- The raw text of what the client said.
- An estimated emotional profile from text analysis:
  - Valence: from -1 (very negative) to +1 (very positive)
  - Arousal: from -1 (very low/flat) to +1 (very activated/agitated)

Use this emotional information to:
- Adjust your empathy (e.g., acknowledge high distress when arousal is high and valence low).
- Choose questions that fit the emotional intensity.
- Still focus on CBT techniques (thoughts, evidence, alternative views).

Important:
- NEVER mention numbers or "valence/arousal" explicitly.
- Talk only in natural emotional language (e.g., "it sounds very overwhelming").
"""

THERAPIST_USER_TEMPLATE_EMO = """
Client just said:
"{client_text}"

Estimated emotion from their words:
- Valence (text): {val_t:.2f}
- Arousal (text): {aro_t:.2f}

Write your next therapist response.
Remember: use this emotional profile internally to guide your tone and focus,
but do NOT mention these scores directly.
"""

# ==============================
# 4) helper เรียก LLM
# ==============================

def chat_once(system_prompt: str, user_prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=512,
    )
    return resp.choices[0].message.content.strip()

# ==============================
# 5) save json, jsonl function
# ==============================

import json
from pathlib import Path

def save_dialogue_json_and_jsonl(turns, base_path: Path):
    """
    base_path เช่น Path('.../baseline/baseline_outputs/dialogue_3_full_baseline')
    จะได้:
      - dialogue_3_full_baseline.json
      - dialogue_3_full_baseline.jsonl
    """
    base_path = Path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)

    json_path = base_path.with_suffix(".json")
    jsonl_path = base_path.with_suffix(".jsonl")

    with json_path.open("w", encoding="utf-8") as f:
        json.dump(turns, f, ensure_ascii=False, indent=2)
    print(f"[SAVE] JSON   -> {json_path}")

    with jsonl_path.open("w", encoding="utf-8") as f:
        for rec in turns:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"[SAVE] JSONL  -> {jsonl_path}")

# ==============================
# 6) main loop – dialogue_emotion-aware
# ==============================

from pathlib import Path

# 1) กำหนด BASE และ METHOD_DIRS ให้เรียบร้อยก่อน
BASE = Path(r"C:\Luna-AI-Therapist\dissonance\craft_dialogue")

METHOD_DIRS = {
    "baseline": BASE / "baseline",
    "emotion": BASE / "emotion",
    "dissonance": BASE / "dissonance",
}

def run_single_dialogue_emotion(dialogue_id: int, max_turns: int = 10):
    out_dir = METHOD_DIRS["emotion"] / "emotion_outputs"
    out_dir.mkdir(parents=True, exist_ok=True)

    problem_seed = PROBLEM_SEEDS[dialogue_id - 1]

    turns = []

    # ---- turn 1: client เริ่ม ----
    first_prompt = CLIENT_USER_TEMPLATE_FIRST.format(problem_seed=problem_seed)
    client_text = chat_once(CLIENT_SYSTEM, first_prompt)
    print(f"CLIENT (t=1): {client_text}\n")

    # ต้องมีตรงนี้ก่อนเอาไป format
    val_t, aro_t = get_text_VA(client_text)
    print(f"[TURN 1] text VA    : val_t={val_t:.3f}, aro_t={aro_t:.3f}")

    therapist_text = chat_once(
        THERAPIST_SYSTEM_EMO,
        THERAPIST_USER_TEMPLATE_EMO.format(
            client_text=client_text,
            val_t=val_t,
            aro_t=aro_t,
        ),
    )
    print(f"THERAPIST (t=1): {therapist_text}\n")

    turns.append({
        "turn": 1,
        "client": client_text,
        "therapist": therapist_text,
        "condition": "emotion_text_only",
        "val_t": val_t,
        "aro_t": aro_t,
    })

    # ---- turns 2..max_turns ----
    for t in range(2, max_turns + 1):
        client_text = chat_once(
            CLIENT_SYSTEM,
            CLIENT_USER_TEMPLATE_NEXT.format(therapist_text=therapist_text),
        )
        print(f"CLIENT (t={t}): {client_text}\n")

        # ต้องคำนวณใหม่ทุก turn
        val_t, aro_t = get_text_VA(client_text)
        print(f"[TURN {t}] text VA    : val_t={val_t:.3f}, aro_t={aro_t:.3f}")

        therapist_text = chat_once(
            THERAPIST_SYSTEM_EMO,
            THERAPIST_USER_TEMPLATE_EMO.format(
                client_text=client_text,
                val_t=val_t,
                aro_t=aro_t,
            ),
        )
        print(f"THERAPIST (t={t}): {therapist_text}\n")

        turns.append({
            "turn": t,
            "client": client_text,
            "therapist": therapist_text,
            "condition": "emotion_text_only",
            "val_t": val_t,
            "aro_t": aro_t,
        })

    base_name = f"dialogue_{dialogue_id}_full_emotion_online"
    base_path = out_dir / base_name
    save_dialogue_json_and_jsonl(turns, base_path)

# Loop run 10 dialogues
if __name__ == "__main__":
    NUM_DIALOGUES = 10
    MAX_TURNS = 10

    for i in range(1, NUM_DIALOGUES + 1):

        print(f"\n=== EMOTION (text-only) dialogue {i} ===")
        run_single_dialogue_emotion(dialogue_id=i, max_turns=MAX_TURNS)


# # Loop run 1 dialogue
# if __name__ == "__main__":
#     run_dialogue_emotion(max_turns=10)



=== EMOTION (text-only) dialogue 1 ===
CLIENT (t=1): Hi Luna, I've been feeling really overwhelmed with work lately. The stress seems to pile up, and I constantly have this nagging fear that I'm going to fail at my job. It feels like no matter how much I try to keep up, I'm always falling short. Honestly, I'm starting to wonder if therapy can really help me deal with this.

[TURN 1] text VA    : val_t=-0.314, aro_t=0.214
THERAPIST (t=1): Hi there! It sounds like you’re really feeling the weight of this overwhelming stress at work, and I can understand how that nagging fear of failure can be incredibly unsettling. It’s tough when it feels like no matter how hard you try, you’re not quite meeting the expectations you set for yourself. 

I want you to know that it’s completely valid to feel this way, and questioning whether therapy can help is a natural part of the process. Can we explore some of those thoughts you have about falling short? What specific situations make you feel this way